# Laboratorio AWS — Predicción de temperatura de un transformador
## Notebook 04 — Inferencia y preparación para despliegue

### Objetivo

Tomar el mejor modelo producido en el Notebook 02 y demostrar el flujo de inferencia.

La prioridad es que el laboratorio funcione aunque no exista un Training Job administrado.

### Flujo principal ejecutable

```text
S3 / model / mejor_modelo.joblib
              ↓
        Descargar modelo
              ↓
       Cargar con joblib
              ↓
     Preparar una observación
              ↓
          Predicción
              ↓
     Evaluación de escenarios
```

### Flujo opcional de nube

```text
Modelo
  ↓
ModelBuilder
  ↓
Endpoint Serverless
  ↓
Inferencia
```

La creación del endpoint queda separada porque sí genera recursos de SageMaker.

## Paso 1. Librerías y configuración

In [ ]:
import json
from pathlib import Path

import boto3
import joblib
import pandas as pd
import numpy as np

import sagemaker

print("Librerías cargadas correctamente.")

## Paso 2. Variables del proyecto

In [ ]:
BUCKET = "siemens-ml-lab-aiarch-2026"
REGION = boto3.Session().region_name or "us-east-2"

MODEL_KEY = "model/mejor_modelo.joblib"
LOCAL_MODEL = Path("mejor_modelo.joblib")

s3 = boto3.client("s3", region_name=REGION)

print(f"Bucket: {BUCKET}")
print(f"Región: {REGION}")
print(f"Modelo: s3://{BUCKET}/{MODEL_KEY}")

## Paso 3. Recuperar el mejor modelo desde S3

Este paso utiliza directamente el artefacto generado por el Notebook 02.

No depende del Training Job del Notebook 03.

In [ ]:
s3.download_file(
    BUCKET,
    MODEL_KEY,
    str(LOCAL_MODEL)
)

model = joblib.load(LOCAL_MODEL)

print("Modelo descargado y cargado correctamente.")
print(f"Tipo de modelo: {type(model).__name__}")

## Paso 4. Preparar una observación para inferencia

Las variables deben mantener exactamente el mismo esquema utilizado durante el entrenamiento.

Se utilizará un escenario representativo de operación de un transformador.

In [ ]:
sample = pd.DataFrame([{
    "temperatura_ambiente": 25.0,
    "corriente": 500.0,
    "carga": 75.0,
    "humedad": 60.0,
    "vibracion": 2.5,
    "potencia": 375.0,
    "horas_operacion": 5000.0
}])

sample

## Paso 5. Realizar la predicción

La salida representa la temperatura estimada del transformador para las condiciones suministradas.

In [ ]:
prediction = float(model.predict(sample)[0])

print(f"Temperatura estimada del transformador: {prediction:.2f}")

## Paso 6. Probar varios escenarios operativos

La inferencia puede utilizarse para analizar escenarios.

No estamos reentrenando el modelo: simplemente cambiamos las condiciones de entrada y observamos la predicción.

In [ ]:
scenarios = pd.DataFrame([
    {
        "escenario": "Operación normal",
        "temperatura_ambiente": 25,
        "corriente": 500,
        "carga": 75,
        "humedad": 60,
        "vibracion": 2.5,
        "potencia": 375,
        "horas_operacion": 5000
    },
    {
        "escenario": "Mayor corriente",
        "temperatura_ambiente": 25,
        "corriente": 650,
        "carga": 85,
        "humedad": 60,
        "vibracion": 2.5,
        "potencia": 552.5,
        "horas_operacion": 5000
    },
    {
        "escenario": "Ambiente elevado",
        "temperatura_ambiente": 35,
        "corriente": 550,
        "carga": 80,
        "humedad": 65,
        "vibracion": 2.8,
        "potencia": 440,
        "horas_operacion": 7000
    }
])

feature_columns = [
    "temperatura_ambiente", "corriente", "carga",
    "humedad", "vibracion", "potencia", "horas_operacion"
]

scenarios["temperatura_predicha"] = model.predict(
    scenarios[feature_columns]
)

display(scenarios[["escenario", "temperatura_predicha"]])

## Paso 7. Cierre y opción Serverless

El flujo principal de inferencia ya está completo.

La siguiente arquitectura es posible:

```text
S3
 ↓
ModelBuilder
 ↓
Serverless Endpoint
 ↓
API de inferencia
```

Sin embargo, la creación del endpoint es **opcional** para este laboratorio.

Si la cuenta no permite desplegar Serverless, no se bloquea el curso: la inferencia local ya demuestra el concepto de serving y el Notebook 05 continuará con Model Registry.

### Resultado

Tenemos un modelo reutilizable y una inferencia reproducible sin depender de infraestructura de entrenamiento.